# 05 — Preprocessing and Feature Engineering

## 1 — Objective

**IT3051 – Fundamentals of Data Mining · Mini Project 2026**

Prepare a reproducible primary modelling dataset and train-fitted sklearn transformer using the approved preprocessing policy. Apply justified row rules, six feature exclusions, and five deterministic features on copies. Then separate `is_canceled`, create one stratified, group-aware global holdout, and learn preprocessing statistics from training predictors only.

The raw CSV remains immutable. No predictive classifier, prediction metrics, resampling, or tuning is used. The City Hotel versus Resort Hotel investigation continues in model development; separate models are not assumed to be superior.

## 2 — Load raw data and reusable code

Use the repository-relative path strategy from the earlier notebooks. Import the reusable implementation rather than hiding cleaning and engineering inside notebook-only code. The file hash and an untouched DataFrame snapshot support final integrity checks.

In [1]:
from pathlib import Path
import sys
import hashlib
import pandas as pd
import numpy as np
import sklearn
from scipy import sparse
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.exceptions import NotFittedError
from sklearn.utils.validation import check_is_fitted
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
project_root = next((candidate for candidate in (cwd, cwd.parent)
    if (candidate / "requirements.txt").is_file() and (candidate / "notebooks").is_dir()
    and (candidate / "data" / "raw").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Run from the repository root or notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.preprocessing import (
    TARGET, DIRECT_LEAKAGE_FEATURES, TEMPORAL_EXCLUSIONS, QUALITY_EXCLUSIONS,
    PRIMARY_SOURCE_EXCLUSIONS, EXCLUDED_FEATURES, ENGINEERED_FEATURES,
    NUMERICAL_FEATURES, CATEGORICAL_FEATURES, engineer_features, clean_rows,
    prepare_training_table, split_features_target, create_predictor_groups, build_preprocessor,
)

data_path = project_root / "data" / "raw" / "hotel_bookings.csv"
if not data_path.is_file():
    raise FileNotFoundError("Place hotel_bookings.csv inside data/raw/.")
raw_hash_before = hashlib.sha256(data_path.read_bytes()).hexdigest()
df = pd.read_csv(data_path)
raw_snapshot = df.copy(deep=True)
print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns from data/raw/hotel_bookings.csv")
print(f"pandas {pd.__version__}; sklearn {sklearn.__version__}")

def markdown_table(table):
    def text(value):
        if pd.isna(value): return "N/A"
        return f"{value:.3f}" if isinstance(value, float) else str(value)
    lines = ["| " + " | ".join(table.columns) + " |", "| " + " | ".join("---" for _ in table.columns) + " |"]
    lines += ["| " + " | ".join(text(v) for v in row) + " |" for row in table.itertuples(index=False, name=None)]
    return "\n".join(lines)

Loaded 119,390 rows × 32 columns from data/raw/hotel_bookings.csv
pandas 3.0.3; sklearn 1.9.0


## 3 — Preprocessing decision plan

Connect the earlier audits to the approved actions. The evidence below is recalculated only where needed to justify this stage; it does not repeat visual EDA. Timing/source exclusions remain distinct from direct leakage and ordinary feature selection.

In [2]:
raw_guest_total = df["adults"] + df["children"] + df["babies"]
raw_stay_total = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
undefined_counts = {column: int(df[column].eq("Undefined").sum()) for column in ["meal", "market_segment", "distribution_channel"]}
decision_rows = [
    ["reservation_status", "Outcome-status relationship established in notebook 04", "Exclude", "Direct target leakage"],
    ["reservation_status_date", "Date of final status; notebook 04", "Exclude", "Direct/outcome leakage"],
    ["assigned_room_type", "Ultimate allocation can reflect later operations", "Exclude", "Strong temporal concern"],
    ["booking_changes", "Count accumulates through check-in/cancellation", "Exclude", "Strong temporal concern"],
    ["company", f"{df['company'].isna().sum():,} missing ({100*df['company'].isna().mean():.3f}%)", "Exclude from primary set", "Extreme missingness and identifier-like sparsity; not leakage"],
    ["adr", f"Range {df['adr'].min():g}–{df['adr'].max():g}; unresolved source/timing", "Exclude from primary set", "Source/timing limitation; optional sensitivity experiment later; not direct leakage"],
    ["deposit_type", "Payment-based category; timing varies", "Retain as categorical", "Document assessment-time limitation"],
    ["days_in_waiting_list", "Known after confirmation; assessment-point dependent", "Retain as numerical", "Timing limitation, not direct leakage"],
    ["agent", f"{df['agent'].nunique():,} observed codes; {df['agent'].isna().sum():,} missing", "Categorical string code; neutral Missing imputation", "Codes are not continuous measurements; missing does not definitely mean No Agent"],
    ["children", f"{df['children'].isna().sum():,} missing", "Train-fitted median imputation", "Do not infer missing children are zero"],
    ["country", f"{df['country'].isna().sum():,} missing", "Categorical Missing imputation", "Retain with neutral missing label"],
    ["Exact duplicates", f"{df.duplicated(keep='first').sum():,} extra full-row copies", "Retain ambiguous copies; group identical predictor profiles at holdout split", "No booking ID establishes identity; blanket removal changes booking frequencies"],
    ["Zero guests", f"{raw_guest_total.eq(0).sum():,} evaluable raw rows", "Remove definitive zeros directly from the raw working copy", "No recorded guests is outside intended input case; uncertain totals are retained"],
    ["Zero stay", f"{raw_stay_total.eq(0).sum():,} raw rows", "No zero-stay-specific removal", "Insufficient evidence all zero-night stays are invalid"],
    ["IQR outliers", "Earlier audit found zero IQR in sparse/count fields", "No IQR filtering or capping", "Screening flags are not evidence of invalidity"],
    ["Undefined categories", str(undefined_counts), "Retain observed labels", "Not automatically missing or erroneous"],
]
decision_table = pd.DataFrame(decision_rows, columns=["Issue / Feature", "Observed Evidence", "Decision", "Reason"])
with pd.option_context("display.max_colwidth", 100):
    display(decision_table)

,Issue / Feature,Observed Evidence,Decision,Reason
0,reservation_status,Outcome-status relationship established in notebook 04,Exclude,Direct target leakage
1,reservation_status_date,Date of final status; notebook 04,Exclude,Direct/outcome leakage
2,assigned_room_type,Ultimate allocation can reflect later operations,Exclude,Strong temporal concern
3,booking_changes,Count accumulates through check-in/cancellation,Exclude,Strong temporal concern
4,company,"112,593 missing (94.307%)",Exclude from primary set,Extreme missingness and identifier-like sparsity; not leakage
5,adr,Range -6.38–5400; unresolved source/timing,Exclude from primary set,Source/timing limitation; optional sensitivity experiment later; not direct leakage
6,deposit_type,Payment-based category; timing varies,Retain as categorical,Document assessment-time limitation
7,days_in_waiting_list,Known after confirmation; assessment-point dependent,Retain as numerical,"Timing limitation, not direct leakage"
8,agent,"333 observed codes; 16,340 missing",Categorical string code; neutral Missing imputation,Codes are not continuous measurements; missing does not definitely mean No Agent
9,children,4 missing,Train-fitted median imputation,Do not infer missing children are zero


## 4 — Create a modelling working copy

Preserve the raw DataFrame and row indices. Source indices identify rows for reproducibility, not real booking identities; the dataset has no unique booking ID.

In [3]:
working = df.copy(deep=True)
assert working is not df and working.index.is_unique
print("Working-copy shape:", working.shape)

Working-copy shape: (119390, 32)


## 5 — Duplicate ambiguity and holdout strategy

Detect exact copies but do not remove them from the primary modelling dataset. Without a unique booking identifier, their identity is ambiguous: they cannot be proved either accidental copies or valid separate reservations.

The hypothetical full-deduplication tables below demonstrate how blanket removal changes the target and hotel distributions. They are diagnostic evidence only and are not used for modelling. The actual working dataset retains all rows except definitive zero-guest cases.

Full-row duplicate removal and predictor-profile separation are different problems. Raw records can differ only in excluded fields and still become identical model inputs. A predictor-only grouping split will prevent identical unencoded inputs crossing the holdout boundary while retaining booking-frequency information. It cannot establish customer-level or true booking-identity independence.

In [4]:
extra_copy_mask = working.duplicated(keep="first")
repeated_row_mask = working.duplicated(keep=False)
raw_pattern_frequencies = working.value_counts(dropna=False)
duplicate_copies_detected = int(extra_copy_mask.sum())
repeated_groups = int(raw_pattern_frequencies.gt(1).sum())
repeated_rows = int(repeated_row_mask.sum())
# Diagnostic subset ONLY: never passed to row cleaning, engineering, or splitting.
hypothetical_unique = working.loc[~extra_copy_mask]

def distribution_comparison(before, after, column, after_label):
    before_counts = before[column].value_counts(dropna=False)
    after_counts = after[column].value_counts(dropna=False)
    return pd.DataFrame({"Raw count": before_counts, "Raw percentage": 100*before_counts/len(before),
        f"{after_label} count": after_counts, f"{after_label} percentage": 100*after_counts/len(after)})

target_hypothetical_comparison = distribution_comparison(working, hypothetical_unique, TARGET, "Hypothetical deduplicated").rename(index={0: "Not Cancelled", 1: "Cancelled"})
hotel_hypothetical_comparison = distribution_comparison(working, hypothetical_unique, "hotel", "Hypothetical deduplicated")
duplicate_target_counts = working.loc[repeated_row_mask, TARGET].value_counts().sort_index()
duplicate_target_summary = pd.DataFrame({"Repeated-group rows": duplicate_target_counts,
    "Percentage within repeated-group rows": 100*duplicate_target_counts/repeated_rows}).rename(index={0: "Not Cancelled", 1: "Cancelled"})
duplicate_summary = pd.DataFrame([{"Extra exact copies detected": duplicate_copies_detected,
    "Repeated full-row groups": repeated_groups, "All rows in repeated groups": repeated_rows,
    "Duplicates removed from primary data": 0}])
display(duplicate_summary)
display(duplicate_target_summary.round(3))
display(target_hypothetical_comparison.round(3))
display(hotel_hypothetical_comparison.round(3))
print("Hypothetical full deduplication is not used in the primary pipeline.")

,Extra exact copies detected,Repeated full-row groups,All rows in repeated groups,Duplicates removed from primary data
0,31994,8171,40165,0


,Repeated-group rows,Percentage within repeated-group rows
is_canceled,,
Not Cancelled,16714,41.613
Cancelled,23451,58.387


,Raw count,Raw percentage,Hypothetical deduplicated count,Hypothetical deduplicated percentage
is_canceled,,,,
Not Cancelled,75166,62.958,63371,72.51
Cancelled,44224,37.042,24025,27.49


,Raw count,Raw percentage,Hypothetical deduplicated count,Hypothetical deduplicated percentage
hotel,,,,
City Hotel,79330,66.446,53428,61.133
Resort Hotel,40060,33.554,33968,38.867


Hypothetical full deduplication is not used in the primary pipeline.


## 6 — Definitively zero-guest treatment

Start from the complete raw working copy, retaining duplicate-looking rows. Remove a row only if adults + children + babies is known and exactly zero. Missing children leave the total unknown and cannot trigger this filter. Zero-stay records and IQR outliers have no separate removal rule; a zero-stay row may still meet the zero-guest rule.

In [5]:
cleaned, row_counts = clean_rows(working)
working_guests = working["adults"] + working["children"] + working["babies"]
zero_guest_mask = working_guests.notna() & working_guests.eq(0)
pd.testing.assert_frame_equal(cleaned, working.loc[~zero_guest_mask])
assert row_counts["duplicates_removed"] == 0
assert row_counts["duplicate_copies_detected"] == duplicate_copies_detected
assert row_counts["final_rows"] == len(working) - int(zero_guest_mask.sum())
assert cleaned.loc[cleaned["children"].isna()].index.equals(working.loc[working["children"].isna()].index)
zero_stays_retained = int((cleaned["stays_in_weekend_nights"] + cleaned["stays_in_week_nights"]).eq(0).sum())
target_filter_comparison = distribution_comparison(working, cleaned, TARGET, "After zero-guest filter").rename(index={0: "Not Cancelled", 1: "Cancelled"})
hotel_filter_comparison = distribution_comparison(working, cleaned, "hotel", "After zero-guest filter")
display(pd.DataFrame([row_counts]))
display(target_filter_comparison.round(3))
display(hotel_filter_comparison.round(3))
print(f"Zero-night rows remaining: {zero_stays_retained:,}; no zero-stay-specific filter applied.")

,raw_rows,duplicate_copies_detected,duplicates_removed,zero_guest_rows_removed,final_rows
0,119390,31994,0,180,119210


,Raw count,Raw percentage,After zero-guest filter count,After zero-guest filter percentage
is_canceled,,,,
Not Cancelled,75166,62.958,75011,62.923
Cancelled,44224,37.042,44199,37.077


,Raw count,Raw percentage,After zero-guest filter count,After zero-guest filter percentage
hotel,,,,
City Hotel,79330,66.446,79163,66.406
Resort Hotel,40060,33.554,40047,33.594


Zero-night rows remaining: 645; no zero-stay-specific filter applied.


## 7 — Feature exclusions

Apply six primary-set exclusions with separate reasons: two direct leakage fields, two strong temporal concerns, company for quality/selection, and ADR for unresolved source/timing. ADR is not labelled direct leakage and may be reconsidered in a later sensitivity experiment. The raw data retains all fields.

In [6]:
exclusion_table = pd.DataFrame([
    *[{"Feature": feature, "Reason": "Direct leakage"} for feature in DIRECT_LEAKAGE_FEATURES],
    *[{"Feature": feature, "Reason": "Strong temporal concern"} for feature in TEMPORAL_EXCLUSIONS],
    *[{"Feature": feature, "Reason": "Extreme missingness / identifier-like sparsity"} for feature in QUALITY_EXCLUSIONS],
    *[{"Feature": feature, "Reason": "Primary-set source/timing exclusion; sensitivity experiment later"} for feature in PRIMARY_SOURCE_EXCLUSIONS],
])
display(exclusion_table)
training_table = prepare_training_table(df)
assert not set(EXCLUDED_FEATURES) & set(training_table.columns)
assert training_table.index.equals(cleaned.index)
print(f"Working modelling table: {training_table.shape}; target retained until separation.")

,Feature,Reason
0,reservation_status,Direct leakage
1,reservation_status_date,Direct leakage
2,assigned_room_type,Strong temporal concern
3,booking_changes,Strong temporal concern
4,company,Extreme missingness / identifier-like sparsity
5,adr,Primary-set source/timing exclusion; sensitivi...


Working modelling table: (119210, 31); target retained until separation.


## 8 — Deterministic feature engineering

Five features are created by `engineer_features` within `prepare_training_table`. Original component features stay available. No lead-time bin is created. Unknown components propagate to totals; family status is positive if either child/baby count is positive, zero only if both are known zeros, otherwise missing. A zero previous-booking total gives rate 0.0.

In [7]:
engineering_table = pd.DataFrame([
    ["total_stay_nights", "stays_in_weekend_nights + stays_in_week_nights", "Planned/recorded total stay; zero is retained"],
    ["total_guests", "adults + children + babies", "Party size; any missing component leaves the total missing"],
    ["family_booking", "1 if children > 0 or babies > 0; 0 if both are known zeros; otherwise missing", "Recorded child/baby presence; not a claim about relationships"],
    ["previous_booking_total", "previous_cancellations + previous_bookings_not_canceled", "Amount of prior booking history"],
    ["previous_cancellation_rate", "previous_cancellations / previous_booking_total when >0; 0.0 when total=0", "Observed prior cancellation share; unknown history stays missing"],
], columns=["Feature", "Formula / rule", "Purpose / missingness"])
display(engineering_table)
display(training_table[list(ENGINEERED_FEATURES)].head())
assert set(ENGINEERED_FEATURES).issubset(training_table.columns)
assert "lead_time_category" not in training_table.columns
# Conversion is representation only; agent nulls have not been imputed.
assert training_table["agent"].isna().equals(cleaned["agent"].isna())
print("Observed agent-code examples:", training_table.loc[training_table["agent"].notna(), "agent"].head().tolist())

,Feature,Formula / rule,Purpose / missingness
0,total_stay_nights,stays_in_weekend_nights + stays_in_week_nights,Planned/recorded total stay; zero is retained
1,total_guests,adults + children + babies,Party size; any missing component leaves the t...
2,family_booking,1 if children > 0 or babies > 0; 0 if both are...,Recorded child/baby presence; not a claim abou...
3,previous_booking_total,previous_cancellations + previous_bookings_not...,Amount of prior booking history
4,previous_cancellation_rate,previous_cancellations / previous_booking_tota...,Observed prior cancellation share; unknown his...


,total_stay_nights,total_guests,family_booking,previous_booking_total,previous_cancellation_rate
0,0,2.0,0.0,0,0.0
1,0,2.0,0.0,0,0.0
2,1,1.0,0.0,0,0.0
3,1,1.0,0.0,0,0.0
4,2,2.0,0.0,0,0.0


Observed agent-code examples: ['304', '240', '240', '303', '240']


## 9 — Missing-data strategy

No imputer has been fitted yet. Numerical features use training medians followed by training-fitted standardization. Categorical features use the neutral label `Missing` and training-fitted one-hot encoding. Preserve `Undefined` as a source label. Missing agent is not asserted to mean no agent. The following table reports missing values before imputation.

In [8]:
missing_counts = training_table.isna().sum()
missing_rows = []
for feature, count in missing_counts.items():
    if count:
        missing_rows.append({"Feature": feature, "Missing modelling rows": int(count),
            "Strategy": "Training median" if feature in NUMERICAL_FEATURES else "Constant Missing category"})
missing_strategy = pd.DataFrame(missing_rows)
display(missing_strategy)

,Feature,Missing modelling rows,Strategy
0,children,4,Training median
1,country,478,Constant Missing category
2,agent,16280,Constant Missing category
3,total_guests,4,Training median
4,family_booking,4,Training median


## 10 — Predictor/target separation

Separate the label from the 30 unencoded predictors. Hotel remains categorical so the general pipeline and future hotel partitions retain their grouping information.

In [9]:
X, y = split_features_target(training_table)
assert TARGET not in X.columns and not set(EXCLUDED_FEATURES) & set(X.columns)
assert y.isin([0, 1]).all()
assert set(X["hotel"].unique()) == {"City Hotel", "Resort Hotel"}
print("Predictors:", X.shape, "Target:", y.shape)

Predictors:

 (119210, 30) Target: (119210,)


## 11 — Group-aware stratified global holdout

Create group signatures from the final **unencoded predictor table**, after the six exclusions and deterministic engineering but before any learned transformation. `create_predictor_groups(X)` hashes a fixed predictor-column order with `index=False`. Neither the target, excluded fields, nor source index enters the signature. The hash is a grouping identifier only and is never a model feature.

Use the first split from **StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)**: one fold is the test partition and the other four are training. This is one approximately 80/20 holdout, not cross-validation modelling. Target labels guide stratification only, not group creation. Do not search folds or seeds for a preferred result.

Groups may differ in size or contain mixed labels, so exact row counts and class proportions cannot be guaranteed. Report actual sizes/rates and use a transparent two-percentage-point train/test rate-gap check as a sanity check, not a significance test or optimisation objective. Reproducibility assumes unchanged input/order and the recorded pandas/sklearn versions.

Disjoint signatures are checked alongside independent complete-row comparison. A rare hash collision would conservatively co-group different profiles rather than separate identical ones. Profile grouping does not establish customer, time, or actual reservation-identity independence.

In [10]:
predictor_groups = create_predictor_groups(X)
assert predictor_groups.index.equals(X.index)
assert "predictor_group" not in X.columns
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_positions, test_positions = next(splitter.split(X, y, groups=predictor_groups))
X_train, X_test = X.iloc[train_positions].copy(), X.iloc[test_positions].copy()
y_train, y_test = y.iloc[train_positions].copy(), y.iloc[test_positions].copy()
train_groups = predictor_groups.iloc[train_positions]
test_groups = predictor_groups.iloc[test_positions]
group_overlap = len(set(train_groups) & set(test_groups))
assert group_overlap == 0, "STOP: predictor groups cross the holdout boundary."
assert X_train.index.intersection(X_test.index).empty
assert len(X_train) + len(X_test) == len(X)
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
for partition in [X_train, X_test]:
    assert set(partition["hotel"].unique()) == {"City Hotel", "Resort Hotel"}

# Compare complete unencoded rows independently of the hash signatures.
train_patterns = pd.MultiIndex.from_frame(X_train)
test_patterns = pd.MultiIndex.from_frame(X_test)
shared_predictor_test_rows = int(test_patterns.isin(train_patterns).sum())
assert shared_predictor_test_rows == 0, "STOP: identical predictor rows cross the holdout boundary."

split_summary = pd.DataFrame([
    {"Partition": label, "Records": len(target), "Cancelled": int(target.sum()),
     "Not Cancelled": int(target.eq(0).sum()), "Cancelled percentage": 100*target.mean()}
    for label, target in [("All modelling records", y), ("Train", y_train), ("Test", y_test)]
])
group_summary = pd.DataFrame([{"All predictor groups": predictor_groups.nunique(),
    "Training groups": train_groups.nunique(), "Test groups": test_groups.nunique(),
    "Group overlap": group_overlap, "Identical-predictor test rows in train": shared_predictor_test_rows}])
class_gap_pp = abs(y_train.mean() - y_test.mean()) * 100
assert class_gap_pp <= 2.0, "Review class imbalance between the fixed grouped partitions; do not select another fold silently."
repeat_train, repeat_test = next(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42).split(X, y, groups=predictor_groups))
np.testing.assert_array_equal(train_positions, repeat_train)
np.testing.assert_array_equal(test_positions, repeat_test)
display(split_summary.round(3))
display(group_summary)
print(f"Test share: {100*len(X_test)/len(X):.3f}%; train/test cancellation-rate gap: {class_gap_pp:.3f} percentage points.")
print("Fixed first split is reproducible; both hotels present; predictor-profile overlap is zero.")

,Partition,Records,Cancelled,Not Cancelled,Cancelled percentage
0,All modelling records,119210,44199,75011,37.077
1,Train,95375,35363,60012,37.078
2,Test,23835,8836,14999,37.072


,All predictor groups,Training groups,Test groups,Group overlap,Identical-predictor test rows in train
0,83531,66831,16700,0,0


Test share: 19.994%; train/test cancellation-rate gap: 0.006 percentage points.
Fixed first split is reproducible; both hotels present; predictor-profile overlap is zero.


## 12 — Numerical and categorical groups

Use explicit semantic groups from the source module. Agent is categorical despite its numeric-looking raw codes; hotel remains categorical. Binary history/family indicators enter the numerical branch. Numerical standardization also applies to count/date-number features as specified; future model-specific alternatives belong to later work.

In [11]:
numerical_features = list(NUMERICAL_FEATURES)
categorical_features = list(CATEGORICAL_FEATURES)
feature_groups = pd.DataFrame([
    *[{"Feature": feature, "Branch": "Numerical"} for feature in numerical_features],
    *[{"Feature": feature, "Branch": "Categorical"} for feature in categorical_features],
])
display(feature_groups)
assert not set(numerical_features) & set(categorical_features)
assert set(numerical_features + categorical_features) == set(X.columns)
assert "agent" in categorical_features and "agent" not in numerical_features
assert "hotel" in categorical_features
assert not ({TARGET} | set(EXCLUDED_FEATURES)) & set(numerical_features + categorical_features)

,Feature,Branch
0,lead_time,Numerical
1,arrival_date_year,Numerical
2,arrival_date_week_number,Numerical
3,arrival_date_day_of_month,Numerical
4,stays_in_weekend_nights,Numerical
5,stays_in_week_nights,Numerical
6,adults,Numerical
7,children,Numerical
8,babies,Numerical
9,is_repeated_guest,Numerical


## 13 — Build an unfitted preprocessing transformer

The ColumnTransformer contains a median-imputer/StandardScaler numerical pipeline and a constant-Missing-imputer/OneHotEncoder categorical pipeline. Unknown categories are ignored at transformation, giving all zeros for that field. No target is supplied. Entirely missing training columns are checked; keep_empty_features preserves their schema rather than silently dropping them.

In [12]:
preprocessor = build_preprocessor(X_train)
try:
    check_is_fitted(preprocessor)
except NotFittedError:
    print("Confirmed: newly built transformer is unfitted.")
else:
    raise AssertionError("build_preprocessor must not fit statistics.")
all_missing_training = X_train.columns[X_train.isna().all()].tolist()
print("Entirely missing training columns:", all_missing_training)
assert not all_missing_training, "Review empty-training-column fallback before proceeding with this dataset."
print("Numerical: median imputer -> StandardScaler")
print("Categorical: constant Missing imputer -> OneHotEncoder(handle_unknown='ignore')")

Confirmed: newly built transformer is unfitted.
Entirely missing training columns: []
Numerical: median imputer -> StandardScaler
Categorical: constant Missing imputer -> OneHotEncoder(handle_unknown='ignore')


## 14 — Fit on training predictors only

This is the only fit of the production transformer. The target, full modelling table, and test partition are not passed to it. For later model cross-validation, rebuild/refit the whole preprocessing pipeline within each training fold rather than reusing these fitted statistics.

In [13]:
X_train_transformed = preprocessor.fit_transform(X_train)
numerical_pipeline = preprocessor.named_transformers_["numerical"]
categorical_pipeline = preprocessor.named_transformers_["categorical"]
numeric_imputer = numerical_pipeline.named_steps["imputer"]
scaler = numerical_pipeline.named_steps["scaler"]
encoder = categorical_pipeline.named_steps["encoder"]
np.testing.assert_allclose(numeric_imputer.statistics_, X_train[numerical_features].median().to_numpy())
assert int(scaler.n_samples_seen_) == len(X_train)
fitted_medians_before_test = numeric_imputer.statistics_.copy()
fitted_means_before_test = scaler.mean_.copy()
categories_before_test = [category.copy() for category in encoder.categories_]
print("Fitted numerical medians/scaling and categorical vocabularies using X_train only.")

Fitted numerical medians/scaling and categorical vocabularies using X_train only.


## 15 — Transform the holdout and validate matrices

Apply transform to the test predictors without refitting. Validate stored sparse values rather than densifying large matrices; implicit sparse entries are zeros. Show a small feature-name sample, not the complete one-hot vocabulary. Encoded/scaled matrices are kept in memory and are not written to CSV.

In [14]:
X_test_transformed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()
def finite_matrix(matrix):
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix)
    return bool(np.isfinite(values).all())
assert finite_matrix(X_train_transformed) and finite_matrix(X_test_transformed)
assert X_train_transformed.shape[1] == X_test_transformed.shape[1] == len(feature_names)
assert X_train_transformed.shape[0] == len(X_train) and X_test_transformed.shape[0] == len(X_test)
np.testing.assert_array_equal(numeric_imputer.statistics_, fitted_medians_before_test)
np.testing.assert_array_equal(scaler.mean_, fitted_means_before_test)
for before, after in zip(categories_before_test, encoder.categories_):
    np.testing.assert_array_equal(before, after)
matrix_summary = pd.DataFrame([
    {"Partition": "Train", "Unencoded rows": len(X_train), "Unencoded features": X_train.shape[1],
     "Transformed features": X_train_transformed.shape[1], "Finite / no missing": finite_matrix(X_train_transformed)},
    {"Partition": "Test", "Unencoded rows": len(X_test), "Unencoded features": X_test.shape[1],
     "Transformed features": X_test_transformed.shape[1], "Finite / no missing": finite_matrix(X_test_transformed)},
])
display(matrix_summary)
print("Transformed feature-name sample:", feature_names[:15].tolist())
print("Matrices remain in memory; no processed files saved.")

,Partition,Unencoded rows,Unencoded features,Transformed features,Finite / no missing
0,Train,95375,30,564,True
1,Test,23835,30,564,True


Transformed feature-name sample: ['numerical__lead_time', 'numerical__arrival_date_year', 'numerical__arrival_date_week_number', 'numerical__arrival_date_day_of_month', 'numerical__stays_in_weekend_nights', 'numerical__stays_in_week_nights', 'numerical__adults', 'numerical__children', 'numerical__babies', 'numerical__is_repeated_guest', 'numerical__previous_cancellations', 'numerical__previous_bookings_not_canceled', 'numerical__days_in_waiting_list', 'numerical__required_car_parking_spaces', 'numerical__total_of_special_requests']
Matrices remain in memory; no processed files saved.


## 16 — Hotel-specific partition preparation

Derive City/Resort subsets of the SAME global group-aware holdout. Later hotel-specific models use corresponding training subsets and compare with the general model on the identical hotel-specific holdout bookings. There are no independent hotel splits and no models fitted now.

In [15]:
hotel_split_rows = []
for partition, values in [("Train", X_train), ("Test", X_test)]:
    for hotel in ["City Hotel", "Resort Hotel"]:
        count = int(values["hotel"].eq(hotel).sum())
        hotel_split_rows.append({"Hotel": hotel, "Partition": partition, "Records": count,
            "Percentage of partition": 100*count/len(values)})
hotel_split_summary = pd.DataFrame(hotel_split_rows)
display(hotel_split_summary.round(3))

,Hotel,Partition,Records,Percentage of partition
0,City Hotel,Train,63044,66.101
1,Resort Hotel,Train,32331,33.899
2,City Hotel,Test,16119,67.627
3,Resort Hotel,Test,7716,32.373


## 17 — Validation and edge cases

Check raw-data integrity, unchanged feature policy, zero-guest-only row removal, and deterministic missing-value semantics. Synthetic fixtures verify that duplicate-looking rows are retained and predictor groups do not depend on target, excluded fields, or row index. These fixtures are code tests, not dataset results. The split assertions stop execution if any identical complete predictor row crosses the holdout boundary.

In [16]:
# Domain edge cases: preserve uncertainty while honouring positive child/baby evidence.
fixture = pd.DataFrame({
    "adults": [1, 1, 0, 2], "children": [np.nan, np.nan, 0.0, 2.0], "babies": [0, 1, 0, 0],
    "stays_in_weekend_nights": [0, 1, 0, 1], "stays_in_week_nights": [0, 2, 0, 3],
    "previous_cancellations": [0, 1, 0, np.nan], "previous_bookings_not_canceled": [0, 3, 2, 1],
})
fixture_before = fixture.copy(deep=True)
fixture_result = engineer_features(fixture)
pd.testing.assert_frame_equal(fixture, fixture_before)
np.testing.assert_allclose(fixture_result["family_booking"], [np.nan, 1, 0, 1], equal_nan=True)
np.testing.assert_allclose(fixture_result["total_guests"], [np.nan, np.nan, 0, 4], equal_nan=True)
np.testing.assert_allclose(fixture_result["previous_cancellation_rate"], [0, 0.25, 0, np.nan], equal_nan=True)

# Test duplicate retention, zero-guest-only removal, and missing-agent preservation.
mini = pd.concat([df.iloc[[0]].copy() for _ in range(3)], ignore_index=True)
mini.loc[:, "agent"] = [9.0, 240.0, np.nan]
mini.loc[:, "adults"] = [1, 1, 0]
mini.loc[:, "children"] = [np.nan, 0.0, 0.0]
mini.loc[:, "babies"] = [0, 1, 0]
mini = pd.concat([mini, mini.iloc[[0]]], ignore_index=True)
mini_before = mini.copy(deep=True)
mini_prepared = prepare_training_table(mini)
pd.testing.assert_frame_equal(mini, mini_before)
assert mini_prepared.index.tolist() == [0, 1, 3]
assert mini_prepared["agent"].tolist() == ["9", "240", "9"]
unknown_agent = mini.iloc[[0]].copy()
unknown_agent.loc[:, "agent"] = np.nan
assert prepare_training_table(unknown_agent)["agent"].isna().all()

# A test-only unseen code must transform without adding a learned category.
probe = X_test.iloc[[0]].copy()
probe.loc[:, "agent"] = "UNSEEN_VALIDATION_CODE"
probe_result = preprocessor.transform(probe)
assert finite_matrix(probe_result) and probe_result.shape[1] == len(feature_names)
agent_position = categorical_features.index("agent")
assert "UNSEEN_VALIDATION_CODE" not in encoder.categories_[agent_position]
for feature, learned in zip(categorical_features, encoder.categories_):
    expected = set(X_train.loc[X_train[feature].notna(), feature])
    if X_train[feature].isna().any(): expected.add("Missing")
    assert set(learned) == expected

assert not ({TARGET} | set(EXCLUDED_FEATURES)) & set(X_train.columns)
assert not ({TARGET} | set(EXCLUDED_FEATURES)) & set(X_test.columns)
assert set(ENGINEERED_FEATURES).issubset(X.columns)
assert set(X["hotel"].unique()) == {"City Hotel", "Resort Hotel"}
assert y.isin([0, 1]).all() and X_train.index.intersection(X_test.index).empty
pd.testing.assert_frame_equal(df, raw_snapshot)
pd.testing.assert_frame_equal(df, pd.read_csv(data_path))
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == raw_hash_before
assert set((project_root / "data" / "processed").iterdir()) == {project_root / "data" / "processed" / ".gitkeep"}

# Signature invariance: source index and excluded/target values are irrelevant.
mini_X, _ = split_features_target(mini_prepared)
mini_groups = create_predictor_groups(mini_X)
assert mini_groups.iloc[0] == mini_groups.iloc[2]
pd.testing.assert_series_equal(create_predictor_groups(mini_X.iloc[:, ::-1]), mini_groups)
reindexed = mini_X.copy()
reindexed.index = [101, 202, 303]
np.testing.assert_array_equal(create_predictor_groups(reindexed).to_numpy(), mini_groups.to_numpy())
changed_outcome = mini.copy(deep=True)
changed_outcome[TARGET] = 1 - changed_outcome[TARGET]
changed_outcome["reservation_status"] = "Synthetic outcome-only change"
changed_outcome["adr"] = 12345.0
changed_X, _ = split_features_target(prepare_training_table(changed_outcome))
pd.testing.assert_series_equal(create_predictor_groups(changed_X), mini_groups)
for forbidden_column in [TARGET, *EXCLUDED_FEATURES]:
    contaminated = mini_X.copy()
    contaminated[forbidden_column] = 0
    try:
        create_predictor_groups(contaminated)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Grouping accepted forbidden field: {forbidden_column}")

assert group_overlap == 0 and shared_predictor_test_rows == 0
assert row_counts["duplicates_removed"] == 0
print("Verified zero group overlap and zero identical unencoded predictor rows across partitions.")
print("All integrity, schema, grouped-split, training-only fitting, and edge-case checks passed. No classifier was trained.")

Verified zero group overlap and zero identical unencoded predictor rows across partitions.
All integrity, schema, grouped-split, training-only fitting, and edge-case checks passed. No classifier was trained.


## 18 — Final preprocessing summary

The following report text is generated from this execution. It supplies the Evaluation 1 findings file, so counts and proportions remain tied to the notebook calculations.

In [17]:
row_summary = (f"Started with {len(df):,} rows and {df.shape[1]} columns. Detected {duplicate_copies_detected:,} extra exact full-row copies "
    f"in {repeated_groups:,} repeated groups involving {repeated_rows:,} records. Removed 0 rows merely for duplication. "
    f"Removed {row_counts['zero_guest_rows_removed']:,} definitive zero-guest rows directly from the raw working copy, "
    f"leaving {len(training_table):,} modelling records, including {zero_stays_retained:,} zero-night records.")
display(Markdown(row_summary))
duplicate_limitation = ("No booking ID establishes whether identical records are accidental copies or separate bookings. Their identity is ambiguous. "
    "Full-row deduplication is diagnostic only here: it would materially change the target and hotel distributions. "
    "Profile grouping preserves frequency information while keeping identical unencoded model inputs on one side of the holdout.")
split_description = ("First deterministic split from StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42). "
    "One fold is test and four are training, producing an approximately 80/20 group-aware stratified holdout. "
    "This is not cross-validation modelling, and no alternative fold/seed was selected. "
    "Groups hash only final unencoded predictors, with fixed column order and index=False, before learned transformations. "
    "Neither the target nor any excluded field enters group creation. The hash is not a model feature.")
report_markdown = (
    "# Preprocessing and Feature Engineering Findings\n\n## Starting Dataset\n\n"
    f"`data/raw/hotel_bookings.csv`: {len(df):,} rows × {df.shape[1]} columns. Executed with pandas {pd.__version__} and sklearn {sklearn.__version__}. "
    "Raw data remains unchanged. Calculations come from notebook 05 and src/preprocessing.py.\n\n"
    "## Row Cleaning\n\n" + row_summary + "\n\n" + duplicate_limitation + "\n\n"
    "Actual target distribution before/after zero-guest filtering:\n\n" + markdown_table(target_filter_comparison.reset_index()) + "\n\n"
    "Actual hotel distribution before/after zero-guest filtering:\n\n" + markdown_table(hotel_filter_comparison.reset_index()) + "\n\n"
    "### Why blanket deduplication was rejected\n\n"
    "Hypothetical full-raw-row deduplication (not the primary modelling population):\n\n"
    + markdown_table(target_hypothetical_comparison.reset_index()) + "\n\n"
    + markdown_table(hotel_hypothetical_comparison.reset_index()) + "\n\n"
    "All members of repeated full-row groups have the following observed target distribution:\n\n"
    + markdown_table(duplicate_target_summary.reset_index()) + "\n\n"
    "Unknown guest totals, zero stays, IQR extremes, and Undefined categories have no additional removal rule. A repeated row may still be removed if it independently meets the definitive zero-guest rule.\n\n"
    "## Feature Exclusions\n\n" + markdown_table(exclusion_table) + "\n\n"
    "The six feature exclusions are unchanged. ADR is not direct leakage and remains a possible later sensitivity experiment. Deposit type and waiting duration are retained with timing limitations; agent remains a categorical identifier.\n\n"
    "## Missing-Value Strategy\n\n" + markdown_table(missing_strategy) + "\n\n"
    "Numerical medians come only from X_train. Missing categorical values use neutral Missing, not No Agent. Undefined categories are retained. "
    f"Entirely missing training columns: {len(all_missing_training)}.\n\n"
    "## Feature Engineering\n\n" + markdown_table(engineering_table) + "\n\n"
    "All five deterministic features and missing-value semantics are unchanged. Components remain available; no lead-time categories are added. Independent numerical imputation need not preserve arithmetic identities after imputation.\n\n"
    "## Encoding and Scaling\n\n"
    "Numerical: median SimpleImputer then StandardScaler. Categorical: constant Missing SimpleImputer then OneHotEncoder(handle_unknown='ignore'). "
    f"There are {len(numerical_features)} numerical and {len(categorical_features)} categorical predictors and {len(feature_names)} transformed features. "
    "Both matrices contain finite values only. Matrices and fitted transformers remain in memory; no processed CSV is saved.\n\n"
    + markdown_table(matrix_summary) + "\n\n"
    "## Train/Test Split\n\n" + split_description + "\n\n" + markdown_table(split_summary) + "\n\n"
    f"Test share is {100*len(X_test)/len(X):.3f}%; train/test cancellation-rate gap is {class_gap_pp:.3f} percentage points. "
    "Stratification is approximate because groups cannot be split. The fixed split passes the documented two-percentage-point sanity check; this is not an inferential test.\n\n"
    + markdown_table(group_summary) + "\n\n"
    "Independent complete-row comparison also confirms zero identical unencoded predictor rows across train/test. Split indices are disjoint and reproducible.\n\n"
    "## City Hotel / Resort Hotel Split Preparation\n\n" + markdown_table(hotel_split_summary) + "\n\n"
    "Both hotels occur in both partitions. Later general and hotel-specific models will reuse these SAME holdout bookings; no independent hotel splits are created.\n\n"
    "## Leakage Prevention\n\n"
    "The target is separate; all six approved excluded fields are absent from predictors and signatures. "
    "The transformer is fitted only with X_train after the split, and X_test uses transform without refitting. Assertions check training medians, category vocabularies, unchanged fitted statistics, and zero profile overlap. "
    "Later model validation should preserve predictor groups and refit learned preprocessing within each training fold. No classifier or model metrics were used.\n\n"
    "## Remaining Methodological Limitations\n\n"
    "Profile separation is not proof of customer-level, time-based, or true reservation-identity independence. Duplicate identity remains ambiguous without booking IDs. "
    "The grouping guarantee is for complete unencoded profiles; imputation or unknown-category encoding can make distinct profiles share a transformed representation. "
    "Group allocation only approximately preserves class proportions and holdout size. Reproducibility assumes the same input/order and recorded software versions. "
    "Retrospective snapshots and deposit/waiting-time availability limitations remain.\n\n"
    "## Evaluation 1 Status\n\nPreprocessing and feature engineering are complete under the revised primary policy. Model development has NOT been performed. No classifiers, prediction metrics, resampling, or tuning were used.\n"
)

Started with 119,390 rows and 32 columns. Detected 31,994 extra exact full-row copies in 8,171 repeated groups involving 40,165 records. Removed 0 rows merely for duplication. Removed 180 definitive zero-guest rows directly from the raw working copy, leaving 119,210 modelling records, including 645 zero-night records.

## 19 — Evaluation 1 readiness

The requested preprocessing implementation and validation are complete. Evidence is available in the decision table, row/distribution tables, group-aware split tables, transformed-shape checks, and edge-case assertions. Model development is next. Learned transformations must be refitted within later group-aware training folds; neither holdout performance nor hotel-specific superiority has been established.